# All ML Regression Analyses using 13 Models

In [3]:
import pandas as pd 
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

# Activate the Inline mod
%matplotlib inline

# To ignore/hide the warnings
import warnings
warnings.filterwarnings("ignore")

## Read Data

In [6]:
df = pd.read_excel("sample_data.xlsx")
df.head(3)

,Car_Name,Car_Age,Target,Present_Price,Kms_Driven,Fuel_Type,Seller_Type,Transmission,Owner
0,ritz,10,3.35,5.59,27000,Petrol,Dealer,Manual,0
1,sx4,11,4.75,9.54,43000,Diesel,Dealer,Manual,0
2,ciaz,7,7.25,9.85,6900,Petrol,Dealer,Manual,0


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 301 entries, 0 to 300
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Car_Name       301 non-null    object 
 1   Car_Age        301 non-null    int64  
 2   Target         301 non-null    float64
 3   Present_Price  301 non-null    float64
 4   Kms_Driven     301 non-null    int64  
 5   Fuel_Type      301 non-null    object 
 6   Seller_Type    301 non-null    object 
 7   Transmission   301 non-null    object 
 8   Owner          301 non-null    int64  
dtypes: float64(2), int64(3), object(4)
memory usage: 21.3+ KB


## Unique Values

In [14]:
# For getting detailed information about the dataset;

def get_unique_values(df):
    
    output_data = []

    for col in df.columns:

        # If the number of unique values in the column is less than or equal to 5
        if df.loc[:, col].nunique() <= 5:
            # Get the unique values in the column
            unique_values = df.loc[:, col].unique()
            # Append the column name, number of unique values, unique values, and data type to the output data
            output_data.append([col, df.loc[:, col].nunique(), unique_values, df.loc[:, col].dtype])
        else:
            # Otherwise, append only the column name, number of unique values, and data type to the output data
            output_data.append([col, df.loc[:, col].nunique(),"-", df.loc[:, col].dtype])

    output_df = pd.DataFrame(output_data, columns=['Column Name', 'Number of Unique Values', ' Unique Values ', 'Data Type'])

    return output_df

get_unique_values(df)

,Column Name,Number of Unique Values,Unique Values,Data Type
0,Car_Name,98,-,object
1,Car_Age,16,-,int64
2,Target,156,-,float64
3,Present_Price,147,-,float64
4,Kms_Driven,206,-,int64
5,Fuel_Type,3,"[Petrol, Diesel, CNG]",object
6,Seller_Type,2,"[Dealer, Individual]",object
7,Transmission,2,"[Manual, Automatic]",object
8,Owner,3,"[0, 1, 3]",int64


* There are 98 category at "Car_Name". We can not apply dummy encoding  to this column. 

* Let's drop it.

## Drop

In [17]:
df = df.drop('Car_Name', axis=1)
df.head(2)

,Car_Age,Target,Present_Price,Kms_Driven,Fuel_Type,Seller_Type,Transmission,Owner
0,10,3.35,5.59,27000,Petrol,Dealer,Manual,0
1,11,4.75,9.54,43000,Diesel,Dealer,Manual,0


## Encoding

* We should apply dummy encoding at regression analyses.

In [1]:
# expecially in regression we can use dummy encoding 

In [18]:
# List the categorical columns
categorical_cols = df.select_dtypes(include=['object', 'category']).columns

# Apply dummy encoding with drop_first=True for the categorical columns (Also, original columns were dropped)
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
df.head(2)

,Car_Age,Target,Present_Price,Kms_Driven,Owner,Fuel_Type_Diesel,Fuel_Type_Petrol,Seller_Type_Individual,Transmission_Manual
0,10,3.35,5.59,27000,0,0,1,0,1
1,11,4.75,9.54,43000,0,1,0,0,1


## Handling with null values

In [8]:
df.isna().sum()

Car_Name         0
Car_Age          0
Target           0
Present_Price    0
Kms_Driven       0
Fuel_Type        0
Seller_Type      0
Transmission     0
Owner            0
dtype: int64


* We don't need this code now. 

* I put here in case of you will need in future.

In [9]:
# Fill null values in categorical columns with mode
for col in df.select_dtypes(include=['object', 'category']):
    df[col].fillna(df[col].mode()[0], inplace=True)

In [10]:
# Fill null values in numerical columns with mean
for col in df.select_dtypes(include=['number']):
    df[col].fillna(df[col].mean(), inplace=True)

## Labelling

In [19]:
X = df.drop(columns ="Target")
y = df["Target"]

## Split as Train & Test

In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

## Scalling

Although scaling is not necessary for tree-based algorithms; it is important for Linear Regression, Regulation, KNN and SVM. 

Additionally, we can use scaled data in tree-based algorithms, too. Sometimes, this can improve our models' performance (speed).   


**Machine Learning Methods That Require Scaling**

Scaling is particularly important for algorithms that are based on distance calculations or gradient-based optimization. However, it is not necessary for all machine learning methods. Below is a summary of when scaling is required and when it is not.

**Algorithms That Require Scaling**

These algorithms are sensitive to the scale of features, so scaling is essential:

**K-Nearest Neighbors (KNN):** KNN relies on distance metrics (like Euclidean or Manhattan). Without scaling, features with larger ranges will dominate the distance calculation, leading to biased predictions. Use either StandardScaler or MinMaxScaler.

**Support Vector Machines (SVM):** SVM constructs hyperplanes based on the distance between data points and the decision boundary. Without scaling, it can be difficult to find the optimal hyperplane.

**Logistic Regression:** Logistic regression uses gradient descent for optimization. Different scales across features can slow down or skew the optimization process, affecting model performance.

**Neural Networks (MLP - Multi-Layer Perceptron):** Neural networks update weights using gradient-based optimization. Large differences in feature scales can cause slower convergence and poor performance.

**Principal Component Analysis (PCA):** PCA is based on capturing the variance in the data. Features with larger ranges will contribute more to the variance, overshadowing others, making scaling essential.

**Gradient Boosting Algorithms (XGBoost, LightGBM, CatBoost):** Although tree-based, these algorithms use gradient-based optimization internally. While generally robust, scaling can improve performance and convergence speed.

**Linear Regression:** For linear regression, particularly with regularization techniques like Ridge or Lasso, scaling is necessary. Regularization is sensitive to the magnitude of coefficients, so scaling ensures all features are treated equally.



**-------------Algorithms That Do Not Require Scaling--------------**

These methods are not sensitive to feature scales and therefore do not need scaling:

**Decision Trees (CART, Random Forest):** Decision trees use threshold-based splits, so the scale of features does not affect the way the tree is constructed. Scaling has no impact on the model’s performance.

**Bagging Methods (Random Forest, Bagging):** Since bagging techniques often rely on decision trees, they also do not require scaling. 

**Naive Bayes:** Naive Bayes models work with probabilities based on feature values, which are independent of scale, so no scaling is necessary.

**Tree-based Gradient Boosting Algorithms:** Tree-based methods like XGBoost, LightGBM, and CatBoost use decision trees, which do not rely on feature scaling. However, scaling may help for the optimization part of these models.

**K-Means Clustering:** Although not strictly necessary, K-means uses distance calculations and can benefit from scaling to ensure all features are treated equally.

* BUT, if you use scaled data at these models, it will not cause a problem. 



**------Summary-----**:

**Algorithms that require scaling:** KNN, SVM, Logistic Regression, Neural Networks, PCA, Gradient Boosting, Linear Regression.
**Algorithms that do not require scaling:** Decision Trees, Random Forest, Naive Bayes, Bagging methods.

In general, distance-based and gradient-based methods require scaling, while tree-based methods do not. However, if you are using a pipeline that involves multiple models, it may be useful to scale all features for consistency across the pipeline.




In [21]:
X_train.head(2)

,Car_Age,Present_Price,Kms_Driven,Owner,Fuel_Type_Diesel,Fuel_Type_Petrol,Seller_Type_Individual,Transmission_Manual
184,16,0.75,26000,1,0,1,1,1
132,7,0.95,3500,0,0,1,1,1


In [28]:
# Initialize the MinMaxScaler
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

# Fit the scaler on the training data and transform it
X_train_scaled = scaler.fit_transform(X_train)

In [ ]:
# when we have dummy data , its good to use minmax scaler 

In [29]:
"""
NOT necessary for you!!!

# Convert the scaled NumPy array back into a pandas DataFrame
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)

# Now you can use .head() on the DataFrame
print(X_train_scaled.head(2))"""

    Car_Age  Present_Price  Kms_Driven     Owner  Fuel_Type_Diesel  \
0  0.642857       0.004660    0.051051  0.333333               0.0   
1  0.000000       0.006827    0.006006  0.000000               0.0   

   Fuel_Type_Petrol  Seller_Type_Individual  Transmission_Manual  
0               1.0                     1.0                  1.0  
1               1.0                     1.0                  1.0  


In [30]:
X_test.head(2)

,Car_Age,Present_Price,Kms_Driven,Owner,Fuel_Type_Diesel,Fuel_Type_Petrol,Seller_Type_Individual,Transmission_Manual
177,8,0.57,24000,0,0,1,1,0
289,8,13.60,10980,0,0,1,0,1


In [3]:
# in training data we fit transform 
# but in test data we only transform 

# to prevent data leakage problem 
# Data leakage is when a model accidentally uses information
# from future or test data during training, causing it to appear more accurate than it actually is.

In [31]:
# Transform the scaler on the test data
X_test_scaled = scaler.transform(X_test)

In [32]:
"""
NOT necessary for you!!!

# Convert the scaled NumPy array back into a pandas DataFrame
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

# Now you can use .head() on the DataFrame
print(X_test_scaled.head(2))"""

    Car_Age  Present_Price  Kms_Driven  Owner  Fuel_Type_Diesel  \
0  0.071429       0.002709    0.047047    0.0               0.0   
1  0.071429       0.143910    0.020981    0.0               0.0   

   Fuel_Type_Petrol  Seller_Type_Individual  Transmission_Manual  
0               1.0                     1.0                  0.0  
1               1.0                     0.0                  1.0  


## Modelling

In [44]:
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge, RidgeCV, Lasso, LassoCV
from sklearn.linear_model import ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

In [47]:
# Fit the models
linear = LinearRegression().fit(X_train_scaled, y_train)
ridge=Ridge().fit(X_train_scaled, y_train)
lasso=Lasso().fit(X_train_scaled, y_train)
enet=ElasticNet().fit(X_train_scaled, y_train)
knn=KNeighborsRegressor().fit(X_train_scaled, y_train)
ada=AdaBoostRegressor().fit(X_train_scaled, y_train)
svm=SVR().fit(X_train_scaled, y_train)
mlpc=MLPRegressor().fit(X_train_scaled, y_train) # Multilayer perceptron. ANN model
dtc=DecisionTreeRegressor().fit(X_train_scaled, y_train)
rf=RandomForestRegressor().fit(X_train_scaled, y_train)
xgb=XGBRegressor().fit(X_train_scaled, y_train)
gbm=GradientBoostingRegressor().fit(X_train_scaled, y_train)
lgb=LGBMRegressor().fit(X_train_scaled, y_train) # LightGBM
catbost=CatBoostRegressor().fit(X_train_scaled, y_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000159 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 151
[LightGBM] [Info] Number of data points in the train set: 240, number of used features: 7
[LightGBM] [Info] Start training from score 4.642292
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -in

100:	learn: 1.4148402	total: 123ms	remaining: 1.09s
101:	learn: 1.4035382	total: 124ms	remaining: 1.09s
102:	learn: 1.3896695	total: 125ms	remaining: 1.09s
103:	learn: 1.3788734	total: 126ms	remaining: 1.09s
104:	learn: 1.3698316	total: 127ms	remaining: 1.08s
105:	learn: 1.3585003	total: 128ms	remaining: 1.08s
106:	learn: 1.3530320	total: 129ms	remaining: 1.08s
107:	learn: 1.3415801	total: 130ms	remaining: 1.07s
108:	learn: 1.3286521	total: 131ms	remaining: 1.07s
109:	learn: 1.3169189	total: 133ms	remaining: 1.07s
110:	learn: 1.3070445	total: 134ms	remaining: 1.08s
111:	learn: 1.2981346	total: 136ms	remaining: 1.08s
112:	learn: 1.2905916	total: 137ms	remaining: 1.08s
113:	learn: 1.2814684	total: 138ms	remaining: 1.07s
114:	learn: 1.2739813	total: 139ms	remaining: 1.07s
115:	learn: 1.2636211	total: 140ms	remaining: 1.07s
116:	learn: 1.2545160	total: 142ms	remaining: 1.07s
117:	learn: 1.2441649	total: 144ms	remaining: 1.07s
118:	learn: 1.2359286	total: 145ms	remaining: 1.07s
119:	learn: 

341:	learn: 0.4666582	total: 504ms	remaining: 970ms
342:	learn: 0.4650863	total: 505ms	remaining: 968ms
343:	learn: 0.4638166	total: 506ms	remaining: 965ms
344:	learn: 0.4612173	total: 507ms	remaining: 963ms
345:	learn: 0.4607783	total: 508ms	remaining: 960ms
346:	learn: 0.4593805	total: 509ms	remaining: 957ms
347:	learn: 0.4568300	total: 510ms	remaining: 955ms
348:	learn: 0.4564093	total: 510ms	remaining: 952ms
349:	learn: 0.4559969	total: 511ms	remaining: 950ms
350:	learn: 0.4552410	total: 512ms	remaining: 948ms
351:	learn: 0.4536801	total: 513ms	remaining: 945ms
352:	learn: 0.4521870	total: 514ms	remaining: 942ms
353:	learn: 0.4511775	total: 515ms	remaining: 940ms
354:	learn: 0.4497394	total: 516ms	remaining: 937ms
355:	learn: 0.4474225	total: 517ms	remaining: 935ms
356:	learn: 0.4462616	total: 518ms	remaining: 933ms
357:	learn: 0.4459117	total: 519ms	remaining: 931ms
358:	learn: 0.4451955	total: 520ms	remaining: 928ms
359:	learn: 0.4435767	total: 521ms	remaining: 926ms
360:	learn: 

629:	learn: 0.2698772	total: 899ms	remaining: 528ms
630:	learn: 0.2694900	total: 901ms	remaining: 527ms
631:	learn: 0.2694655	total: 902ms	remaining: 525ms
632:	learn: 0.2693125	total: 903ms	remaining: 524ms
633:	learn: 0.2692378	total: 905ms	remaining: 523ms
634:	learn: 0.2687140	total: 908ms	remaining: 522ms
635:	learn: 0.2679325	total: 910ms	remaining: 521ms
636:	learn: 0.2676401	total: 911ms	remaining: 519ms
637:	learn: 0.2669670	total: 913ms	remaining: 518ms
638:	learn: 0.2665587	total: 914ms	remaining: 516ms
639:	learn: 0.2660998	total: 915ms	remaining: 515ms
640:	learn: 0.2656080	total: 917ms	remaining: 514ms
641:	learn: 0.2652602	total: 919ms	remaining: 512ms
642:	learn: 0.2649154	total: 920ms	remaining: 511ms
643:	learn: 0.2647335	total: 921ms	remaining: 509ms
644:	learn: 0.2643590	total: 922ms	remaining: 508ms
645:	learn: 0.2642074	total: 923ms	remaining: 506ms
646:	learn: 0.2636413	total: 924ms	remaining: 504ms
647:	learn: 0.2634547	total: 925ms	remaining: 503ms
648:	learn: 

870:	learn: 0.1993631	total: 1.28s	remaining: 189ms
871:	learn: 0.1990543	total: 1.28s	remaining: 188ms
872:	learn: 0.1987681	total: 1.28s	remaining: 186ms
873:	learn: 0.1985726	total: 1.28s	remaining: 185ms
874:	learn: 0.1985164	total: 1.28s	remaining: 183ms
875:	learn: 0.1982975	total: 1.28s	remaining: 182ms
876:	learn: 0.1979208	total: 1.28s	remaining: 180ms
877:	learn: 0.1976325	total: 1.29s	remaining: 179ms
878:	learn: 0.1973071	total: 1.29s	remaining: 177ms
879:	learn: 0.1970358	total: 1.29s	remaining: 176ms
880:	learn: 0.1968793	total: 1.29s	remaining: 174ms
881:	learn: 0.1967096	total: 1.29s	remaining: 173ms
882:	learn: 0.1964732	total: 1.29s	remaining: 171ms
883:	learn: 0.1963016	total: 1.29s	remaining: 170ms
884:	learn: 0.1959125	total: 1.29s	remaining: 168ms
885:	learn: 0.1957200	total: 1.29s	remaining: 167ms
886:	learn: 0.1953929	total: 1.3s	remaining: 165ms
887:	learn: 0.1951273	total: 1.3s	remaining: 164ms
888:	learn: 0.1947683	total: 1.3s	remaining: 162ms
889:	learn: 0.1

## Eval Metrics

In [54]:
models=[linear,ridge,lasso,enet,knn,ada,svm,mlpc,dtc,rf,xgb,gbm,lgb,catbost]

# For train;
def ML(y,models):
    r2_score=models.score(X_train_scaled, y_train)
    return r2_score
for i in models:
    print(i,"Algorithm succed rate :", ML("survived",i))

LinearRegression() Algorithm succed rate : 0.8886517300804564
Ridge() Algorithm succed rate : 0.8279940941365418
Lasso() Algorithm succed rate : 0.1444873504727131
ElasticNet() Algorithm succed rate : 0.20082521592084923
KNeighborsRegressor() Algorithm succed rate : 0.857362402680176
AdaBoostRegressor() Algorithm succed rate : 0.9707788675850607
SVR() Algorithm succed rate : 0.610690444665906
MLPRegressor() Algorithm succed rate : 0.643878748188345
DecisionTreeRegressor() Algorithm succed rate : 1.0
RandomForestRegressor() Algorithm succed rate : 0.9837945570263724
XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, gpu_id=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
            

In [55]:
# For test;
def ML(y,models):
    r2_score=models.score(X_test_scaled, y_test)
    return r2_score
for i in models:
    print(i,"Algorithm succed rate :", ML("survived",i))

LinearRegression() Algorithm succed rate : 0.8489813024899066
Ridge() Algorithm succed rate : 0.7837567859000829
Lasso() Algorithm succed rate : 0.1624615665269098
ElasticNet() Algorithm succed rate : 0.22033692156322826
KNeighborsRegressor() Algorithm succed rate : 0.920822980791902
AdaBoostRegressor() Algorithm succed rate : 0.9282008766021559
SVR() Algorithm succed rate : 0.6970730951191091
MLPRegressor() Algorithm succed rate : 0.6366540640978883
DecisionTreeRegressor() Algorithm succed rate : 0.9436464890035525
RandomForestRegressor() Algorithm succed rate : 0.9595271183412645
XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, gpu_id=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=N

* You can choose best models and apply hyperparameter tuning.

* Using best hyperparameters nd all data, you can obtain final model, and save the final model. 

# Hyperparameter Tuning

# Final Model 

## Save the Final Model

In [ ]:
# TASK DONE!!!!!!!